In [44]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import requests
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core import tools
from requests_oauthlib import OAuth1

SAIS_APP_URL = os.getenv("SAIS_APP_URL")
SAIS_TOKEN = os.getenv("SAIS_TOKEN")

if not SAIS_APP_URL or not SAIS_TOKEN:
    raise SystemExit("Error: SAIS_APP_URL and SAIS_TOKEN environment variables must be set.")

PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")
PROXY_ENABLED = os.getenv("PROXY_ENABLED", "false").lower() == "true"
SSL_VERIFY = os.getenv("SSL_VERIFY", "true").lower() == "true"

def generate_answer(context: str, query: str, model: str = None) -> str:
    selected_model = model or "gpt-4"

    response = requests.post(
        f"{SAIS_APP_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {SAIS_TOKEN}",
            "Content-Type": "application/json",
            "ApplicationType": "BRProduct",
        },
        json={
            "model": selected_model,
            "instructions": "You are an AI technical support assistant.\n\nAnswer the user's question to the best of your knowledge.",
            "input": context + "\n\n" + query,
        },
        verify=SSL_VERIFY,
        proxies={
            "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
        }
    )

    if response.status_code == 200:
        response_data = response.json()
        try:
            outputs = response_data["body"]["output"]
            texts = []
            for item in outputs:
                if item.get("type") == "message":
                    for content in item.get("content", []):
                        if content.get("type") == "output_text":
                            texts.append(content.get("text", ""))
            if texts:
                return "\n".join(texts)
            return "Error: No output text found in the response."
        except KeyError:
            return "Error: Unexpected response format."
    else:
        return f"Error: Request failed with status code {response.status_code}"


In [45]:
class Mystate(TypedDict):
    topic : str
    firstdraft: str
    feedback: str
    iteration: int
    isapproved: bool


In [46]:
def textdraftgenerator(state: Mystate) -> dict:
    topic = state["topic"]
    context = f"Write a concise tweet (max 280 characters) about: {topic}. Include relevant hashtags."
    query = "Keep it short, punchy, and under 280 characters. Include at least one hashtag."
    draft = generate_answer(context, query, model="gpt-4o-mini")
    print(f"Generated draft: {draft}")
    return {"firstdraft": draft}

In [47]:
def draftevaluator(state: Mystate) -> dict:
    draft = state["firstdraft"]
    iteration = state.get("iteration", 0) + 1
    
    errors = []
    if len(draft) > 280:
        errors.append("Draft is too long for X.")
    if "#" not in draft:
        errors.append("Missing essential educational hashtags.")
    
    if errors:
        feedback = f"Needs Improvement: {' '.join(errors)}"
        print(f"Iteration {iteration} - {feedback}")
        return {"feedback": feedback, "iteration": iteration, "isapproved": False}
    
    context = f"Evaluate the following draft:\n{draft}"
    query = "Please provide feedback on the draft, if it is good return 'Approved' else return 'Needs Improvement' with suggestions."
    result = generate_answer(context, query, model="gpt-4o-mini")
    approved = "Approved" in result
    feedback = "Approved" if approved else f"Needs Improvement: {result}"
    print(f"Iteration {iteration} - {feedback[:50]}...")
    return {"feedback": feedback, "iteration": iteration, "isapproved": approved}

In [48]:
def optimize_draft(state: Mystate) -> dict:
    draft = state["firstdraft"]
    feedback = state["feedback"]
    context = f"Rewrite this tweet to fix the issues. Keep it under 280 characters with hashtags.\nDraft: {draft}\nFeedback: {feedback}"
    query = "Provide only the improved tweet text, nothing else."
    improved = generate_answer(context, query, model="gpt-4o-mini")
    print(f"Optimized draft: {improved}")
    return {"firstdraft": improved}

In [49]:
# Conditional router: stop after 3 iterations even if not approved
def conditional_router(state: Mystate) -> str:
    if state.get("isapproved"):
        return "Approved"
    if state.get("iteration", 0) >= 3:
        print("⚠️ Max iterations (3) reached. Force publishing current draft.")
        return "Approved"
    return "Needs Improvement"

In [50]:
#tool to call X API
def post_to_x_tool(text_content: str) -> str:
    """
    Independent tool function that authenticates via OAuth 1.0a 
    and publishes text directly to the X v2 API.
    """
    api_key = os.environ.get("X_API_KEY")
    api_secret = os.environ.get("X_API_SECRET")
    access_token = os.environ.get("X_ACCESS_TOKEN")
    access_secret = os.environ.get("X_ACCESS_SECRET")
    
    if not all([api_key, api_secret, access_token, access_secret]):
        return "Error: One or more X API authentication keys are missing from environment variables."

    url = "https://api.x.com/2/tweets"
    
    auth_layer = OAuth1(api_key, api_secret, access_token, access_secret)
    
    headers = {"Content-Type": "application/json"}
    payload = {"text": text_content}
    
    try:
        response = requests.post(
            url, auth=auth_layer, headers=headers, json=payload,
            timeout=10, verify=SSL_VERIFY,
            proxies={
                "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
                "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            }
        )
        
        if response.status_code == 201:
            tweet_id = response.json()["data"]["id"]
            return f"Success! Tweet published. ID: {tweet_id}"
        else:
            return f"Failed {response.status_code}: {response.text}"
            
    except Exception as e:
        return f"Network Exception: {str(e)}"

In [53]:
def post_to_x_tool_without_llm():
    """Standalone test: post a hardcoded tweet to X API."""
    api_key = os.environ.get("X_API_KEY")
    api_secret = os.environ.get("X_API_SECRET")
    access_token = os.environ.get("X_ACCESS_TOKEN")
    access_secret = os.environ.get("X_ACCESS_SECRET")

    if not all([api_key, api_secret, access_token, access_secret]):
        print("Error: One or more X API authentication keys are missing.")
        return

    url = "https://api.x.com/2/tweets"
    auth_layer = OAuth1(api_key, api_secret, access_token, access_secret)

    tweet = "The future of AI in healthcare is bright! From personalized treatment plans to early disease detection, AI can revolutionize patient care and improve outcomes. Let's embrace innovation for a healthier tomorrow! #AIinHealthcare #HealthTech"

    try:
        response = requests.post(
            url, auth=auth_layer,
            headers={"Content-Type": "application/json"},
            json={"text": tweet},
            timeout=10, verify=False
        )
        print(f"Status: {response.status_code}")
        print(f"Response: {response.text}")
    except Exception as e:
        print(f"Error: {e}")

post_to_x_tool_without_llm()

c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 402
Response: {"detail":"credits depleted","status":402,"title":"Payment Required","type":"https://api.x.com/2/problems/credits-depleted"}


In [51]:
def final_publish_node(state: Mystate) -> dict:
    """
    A LangGraph node that reads from the state, runs the tool,
    and updates the state with the result.
    """
    print("🚀 Draft Approved! Publishing to X...")
    
    # Grab the approved text from state
    approved_text = state["firstdraft"]
           
    # Call our independent tool function
    api_result = post_to_x_tool(approved_text)
    
    print(f"Publish result: {api_result}")
    return {"feedback": api_result}

In [52]:
graph = StateGraph(Mystate)

graph.add_node("textdraftgenerator", textdraftgenerator)
graph.add_node("draftevaluator", draftevaluator)
graph.add_node("optimize_draft", optimize_draft)
graph.add_node("final_publish_node", final_publish_node)

graph.add_edge(START, "textdraftgenerator")
graph.add_edge("textdraftgenerator", "draftevaluator")

graph.add_conditional_edges("draftevaluator", conditional_router, {
    "Approved": "final_publish_node",
    "Needs Improvement": "optimize_draft"
})

graph.add_edge("optimize_draft", "draftevaluator")
graph.add_edge("final_publish_node", END)

fin = graph.compile()

initial_state = {
    "topic": "The future of AI in healthcare",
    "firstdraft": "",
    "feedback": "",
    "iteration": 0,
    "isapproved": False
}

result_state = fin.invoke(initial_state)

print("Final State:", result_state)


c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Generated draft: The future of AI in healthcare is bright! 🎉 From personalized treatment plans to early disease detection, AI can revolutionize patient care and improve outcomes. Let's embrace innovation for a healthier tomorrow! 💡🤖 #AIinHealthcare #HealthTech


c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Iteration 1 - Approved...
🚀 Draft Approved! Publishing to X...
Publish result: Network Exception: HTTPSConnectionPool(host='api.x.com', port=443): Max retries exceeded with url: /2/tweets (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))
Final State: {'topic': 'The future of AI in healthcare', 'firstdraft': "The future of AI in healthcare is bright! 🎉 From personalized treatment plans to early disease detection, AI can revolutionize patient care and improve outcomes. Let's embrace innovation for a healthier tomorrow! 💡🤖 #AIinHealthcare #HealthTech", 'feedback': "Network Exception: HTTPSConnectionPool(host='api.x.com', port=443): Max retries exceeded with url: /2/tweets (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 407 Proxy Authentication Required')))", 'iteration': 1, 'isapproved': True}
